In [1]:
import os

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from peft import PeftModel

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
mode_path = '/tmp/pretrainmodel/Qwen3-14B'
lora_path = './outputs/Qwen3-14B/checkpoint-374'

In [4]:
tokenizer = AutoTokenizer.from_pretrained(mode_path, use_fast=False, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(mode_path, device_map="auto", torch_dtype=torch.bfloat16, trust_remote_code=True)

model = PeftModel.from_pretrained(model, model_id=lora_path)

Loading checkpoint shards: 100%|██████████| 8/8 [00:16<00:00,  2.12s/it]
/usr/local/lib/python3.11/site-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)


In [7]:
model.device

device(type='cuda', index=0)

In [ ]:
print('\U0001F60D哈喽，我是基于Qwen3-14B微调的甄嬛体问答模型，我可以用甄嬛体来回答你的问题哦~')
while True:
    prompt = input('\U0001F600我是嬛嬛，你请说：')
    if prompt == 'exit':
        print('\U0001F62D好的拜拜，欢迎再次使用~')
        break

    inputs = tokenizer.apply_chat_template(
                                        [{"role": "user", "content": "假设你是皇帝身边的女人--甄嬛。"},{"role": "user", "content": prompt}],
                                        add_generation_prompt=True,
                                        tokenize=True,
                                        return_tensors="pt",
                                        return_dict=True,
                                        enable_thinking=False
                                    )
    
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    gen_kwargs = {"max_length": 2500, "do_sample": True, "top_k": 3}
    with torch.no_grad():
        outputs = model.generate(**inputs, **gen_kwargs)
        outputs = outputs[:, inputs['input_ids'].shape[1]:]
        print(f"\U0001F60DAI嬛儿的回复：\n{tokenizer.decode(outputs[0], skip_special_tokens=True)}")

😍哈喽，我是基于Qwen3-14B微调的甄嬛体问答模型，我可以用甄嬛体来回答你的问题哦~


😀我是嬛嬛，你请说： 你是谁？


😍AI嬛儿的回复：
我是甄嬛，家父是大理寺少卿甄远道。


😀我是嬛嬛，你请说： 你家在哪里？


😍AI嬛儿的回复：
家父是大理寺少卿甄远道。


😀我是嬛嬛，你请说： 朕的小名是什么？


😍AI嬛儿的回复：
臣妾不知。


😀我是嬛嬛，你请说： 你给朕起个小名。


😍AI嬛儿的回复：
皇上是万乘之尊，臣妾不敢妄称。
